In [4]:
import os, random, numpy as np, tensorflow as tf, pandas as pd

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:

base_dir = "dataset/chest_xray/chest_xray"   # change if needed
trainval_dir = os.path.join(base_dir, "train")  # we'll also include val manually below
val_dir_kaggle = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen_mnet = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    validation_split=0.2,
    rotation_range=5,
    width_shift_range=0.02,
    height_shift_range=0.02,
    zoom_range=0.05,
    brightness_range=(0.9, 1.1),
    horizontal_flip=False
)

val_datagen_mnet = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    validation_split=0.2
)

test_datagen_mnet = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess
)

train_gen_mnet = train_datagen_mnet.flow_from_directory(
    trainval_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="training", shuffle=True
)

val_gen_mnet = val_datagen_mnet.flow_from_directory(
    trainval_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="validation", shuffle=False
)

test_gen_mnet = test_datagen_mnet.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False
)


Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


## MobileNetV2 lightweight model (Stage 1 = head only)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

def build_mobilenetv2(dropout=0.3):
    base = MobileNetV2(weights="imagenet", 
                       include_top=False, 
                       input_shape=IMG_SIZE + (3,))
    base.trainable = False  # Stage 1: freeze

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auroc"),
            tf.keras.metrics.AUC(name="auprc", curve="PR"),
        ],
    )
    return model

mnet = build_mobilenetv2(dropout=0.3)
mnet.summary()


2026-03-31 17:41:03.890865: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-03-31 17:41:03.890894: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-03-31 17:41:03.890903: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2026-03-31 17:41:03.890963: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-31 17:41:03.891007: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_mnet = [
    ModelCheckpoint("best_mobilenetv2.keras", monitor="val_auprc", mode="max",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_auprc", mode="max", patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_auprc", mode="max", factor=0.5,
                      patience=2, min_lr=1e-6, verbose=1),
]


## Stage 1 (head only)

In [ ]:
hist1 = mnet.fit(
    train_gen_mnet,
    validation_data=val_gen_mnet,
    epochs=10,
    callbacks=callbacks_mnet
)


Epoch 1/10


2026-03-31 17:41:07.366683: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 0.8442 - auprc: 0.9496 - auroc: 0.8653 - loss: 0.3560
Epoch 1: val_auprc improved from None to 0.99690, saving model to best_mobilenetv2.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 53s 351ms/step - accuracy: 0.8970 - auprc: 0.9828 - auroc: 0.9528 - loss: 0.2559 - val_accuracy: 0.9626 - val_auprc: 0.9969 - val_auroc: 0.9909 - val_loss: 0.1430 - learning_rate: 0.0010
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 0.9492 - auprc: 0.9935 - auroc: 0.9818 - loss: 0.1558
Epoch 2: val_auprc improved from 0.99690 to 0.99786, saving model to best_mobilenetv2.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 43s 331ms/step - accuracy: 0.9509 - auprc: 0.9950 - auroc: 0.9856 - loss: 0.1429 - val_accuracy: 0.9664 - val_auprc: 0.9979 - val_auroc: 0.9937 - val_loss: 0.1101 - learning_rate: 0.0010
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - accuracy: 0.9560 - auprc: 0.9963 - auroc: 0.9892 - loss: 0.1220
Epoch 3: val_auprc improved from 0.997

## Stage 2 fine-tuning

In [ ]:
mnet.load_weights("best_mobilenetv2.keras")

base = mnet.layers[1]
base.trainable = True

# Unfreeze only the last chunk (light fine-tune)
for layer in base.layers[:-30]:
    layer.trainable = False

mnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auroc"),
        tf.keras.metrics.AUC(name="auprc", curve="PR"),
    ],
)

hist2 = mnet.fit(
    train_gen_mnet,
    validation_data=val_gen_mnet,
    epochs=10,
    callbacks=callbacks_mnet
)


Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step - accuracy: 0.9123 - auprc: 0.9853 - auroc: 0.9646 - loss: 0.2139
Epoch 1: val_auprc did not improve from 0.99896
131/131 ━━━━━━━━━━━━━━━━━━━━ 57s 365ms/step - accuracy: 0.9319 - auprc: 0.9907 - auroc: 0.9763 - loss: 0.1678 - val_accuracy: 0.9751 - val_auprc: 0.9989 - val_auroc: 0.9967 - val_loss: 0.0658 - learning_rate: 1.0000e-05
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.9532 - auprc: 0.9962 - auroc: 0.9886 - loss: 0.1210
Epoch 2: val_auprc improved from 0.99896 to 0.99901, saving model to best_mobilenetv2.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 46s 347ms/step - accuracy: 0.9559 - auprc: 0.9965 - auroc: 0.9897 - loss: 0.1148 - val_accuracy: 0.9751 - val_auprc: 0.9990 - val_auroc: 0.9971 - val_loss: 0.0612 - learning_rate: 1.0000e-05
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - accuracy: 0.9657 - auprc: 0.9978 - auroc: 0.9932 - loss: 0.0927
Epoch 3: val_auprc improved from 0.99901 to 0.99913, saving 

## Threshold validation

In [ ]:
from utils import eval_thresholds


In [ ]:
thresholds = np.linspace(0.2, 0.9, 71)

val_table = eval_thresholds(mnet, val_gen_mnet, thresholds)  # ✅ this returns a DataFrame

TARGET_SENS = 0.96
eligible = val_table[val_table["Sensitivity"] >= TARGET_SENS]
best = eligible.sort_values("Specificity", ascending=False).iloc[0]
best


Threshold       0.770
Accuracy        0.969
Macro F1        0.961
Weighted F1     0.970
Specificity     0.996
Sensitivity     0.960
FN             31.000
FP              1.000
Name: 57, dtype: float64

In [ ]:
thresholds = np.linspace(0.3, 0.9, 61)

val_table = eval_thresholds(mnet, val_gen_mnet, thresholds)
val_table

,Threshold,Accuracy,Macro F1,Weighted F1,Specificity,Sensitivity,FN,FP
0,0.30,0.984,0.979,0.984,0.981,0.985,12,5
1,0.31,0.984,0.979,0.984,0.981,0.985,12,5
2,0.32,0.985,0.980,0.985,0.985,0.985,12,4
3,0.33,0.986,0.981,0.986,0.989,0.985,12,3
4,0.34,0.986,0.981,0.986,0.989,0.985,12,3
...,...,...,...,...,...,...,...,...
56,0.86,0.962,0.952,0.962,1.000,0.948,40,0
57,0.87,0.960,0.950,0.961,1.000,0.946,42,0
58,0.88,0.958,0.947,0.959,1.000,0.943,44,0
59,0.89,0.953,0.942,0.954,1.000,0.937,49,0


In [ ]:
from utils import evaluate


In [ ]:
# Screening mode
evaluate(mnet, test_gen_mnet, name="MobileNetV2 Screening", threshold=0.33)

# Diagnostic mode
evaluate(mnet, test_gen_mnet, name="MobileNetV2 Diagnostic", threshold=0.86)



=== MobileNetV2 Screening @ threshold 0.330 ===
[[134 100]
 [  4 386]]
              precision    recall  f1-score   support

      NORMAL       0.97      0.57      0.72       234
   PNEUMONIA       0.79      0.99      0.88       390

    accuracy                           0.83       624
   macro avg       0.88      0.78      0.80       624
weighted avg       0.86      0.83      0.82       624

AUROC: 0.9666 | AUPRC: 0.9765
Macro F1: 0.8009 | Weighted F1: 0.8210
Sensitivity: 0.9897 | Specificity: 0.5726 | Precision: 0.7942

=== MobileNetV2 Diagnostic @ threshold 0.860 ===
[[170  64]
 [  8 382]]
              precision    recall  f1-score   support

      NORMAL       0.96      0.73      0.83       234
   PNEUMONIA       0.86      0.98      0.91       390

    accuracy                           0.88       624
   macro avg       0.91      0.85      0.87       624
weighted avg       0.89      0.88      0.88       624

AUROC: 0.9666 | AUPRC: 0.9765
Macro F1: 0.8696 | Weighted F1: 0.8806
S

{'Model': 'MobileNetV2 Diagnostic',
 'Flip': None,
 'Class Weights': 'None',
 'Threshold': 0.86,
 'AUROC': 0.9665625684856454,
 'AUPRC': 0.9765428436397701,
 'Accuracy': 0.8846153846153846,
 'Macro F1': 0.8695591582663631,
 'Weighted F1': 0.8806382682213035,
 'Sensitivity': 0.9794871794871794,
 'Specificity': 0.7264957264957265,
 'Precision': 0.8565022421524664,
 'TN': 170,
 'FP': 64,
 'FN': 8,
 'TP': 382}

Despite its significantly smaller size, MobileNetV2 achieved AUROC and AUPRC comparable to DenseNet121. In screening mode, MobileNetV2 achieved the highest sensitivity (98.97%) with only four missed pneumonia cases, while in a higher-specificity operating mode it reduced false negatives by nearly 50% compared to the baseline CNN. These results demonstrate that lightweight architectures can achieve clinically competitive performance when combined with appropriate threshold selection

In [ ]:
# MobileNetV2 — Screening operating point
mobilenet_screen_res = evaluate(
    mnet,
    test_gen_mnet,
    name="MobileNetV2",
    threshold=0.33
)

mobilenet_screen_res["Mode"] = "Screening"
mobilenet_screen_res["Params"] = mnet.count_params()

mobilenet_screen_res



=== MobileNetV2 @ threshold 0.330 ===
[[134 100]
 [  4 386]]
              precision    recall  f1-score   support

      NORMAL       0.97      0.57      0.72       234
   PNEUMONIA       0.79      0.99      0.88       390

    accuracy                           0.83       624
   macro avg       0.88      0.78      0.80       624
weighted avg       0.86      0.83      0.82       624

AUROC: 0.9666 | AUPRC: 0.9765
Macro F1: 0.8009 | Weighted F1: 0.8210
Sensitivity: 0.9897 | Specificity: 0.5726 | Precision: 0.7942


{'Model': 'MobileNetV2',
 'Flip': None,
 'Class Weights': 'None',
 'Threshold': 0.33,
 'AUROC': 0.9665625684856454,
 'AUPRC': 0.9765428436397701,
 'Accuracy': 0.8333333333333334,
 'Macro F1': 0.8008543231698335,
 'Weighted F1': 0.8209603770805715,
 'Sensitivity': 0.9897435897435898,
 'Specificity': 0.5726495726495726,
 'Precision': 0.7942386831275721,
 'TN': 134,
 'FP': 100,
 'FN': 4,
 'TP': 386,
 'Mode': 'Screening',
 'Params': 2259265}

In [ ]:
# MobileNetV2 — Diagnostic operating point
mobilenet_diag_res = evaluate(
    mnet,
    test_gen_mnet,
    name="MobileNetV2",
    threshold=0.86
)

mobilenet_diag_res["Mode"] = "Diagnostic"
mobilenet_diag_res["Params"] = mnet.count_params()

mobilenet_diag_res



=== MobileNetV2 @ threshold 0.860 ===
[[170  64]
 [  8 382]]
              precision    recall  f1-score   support

      NORMAL       0.96      0.73      0.83       234
   PNEUMONIA       0.86      0.98      0.91       390

    accuracy                           0.88       624
   macro avg       0.91      0.85      0.87       624
weighted avg       0.89      0.88      0.88       624

AUROC: 0.9666 | AUPRC: 0.9765
Macro F1: 0.8696 | Weighted F1: 0.8806
Sensitivity: 0.9795 | Specificity: 0.7265 | Precision: 0.8565


{'Model': 'MobileNetV2',
 'Flip': None,
 'Class Weights': 'None',
 'Threshold': 0.86,
 'AUROC': 0.9665625684856454,
 'AUPRC': 0.9765428436397701,
 'Accuracy': 0.8846153846153846,
 'Macro F1': 0.8695591582663631,
 'Weighted F1': 0.8806382682213035,
 'Sensitivity': 0.9794871794871794,
 'Specificity': 0.7264957264957265,
 'Precision': 0.8565022421524664,
 'TN': 170,
 'FP': 64,
 'FN': 8,
 'TP': 382,
 'Mode': 'Diagnostic',
 'Params': 2259265}

In [ ]:
import pandas as pd
from pathlib import Path

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

pd.DataFrame([mobilenet_screen_res, mobilenet_diag_res]).to_csv(
    out_dir / "mobilenet_results.csv",
    index=False
)


In [ ]:
print("Screening FN/FP:", mobilenet_screen_res["FN"], mobilenet_screen_res["FP"])
print("Diagnostic FN/FP:", mobilenet_diag_res["FN"], mobilenet_diag_res["FP"])


Screening FN/FP: 4 100
Diagnostic FN/FP: 8 64
